# DD2360 Assignment 2
Runtime → Change runtime type → **T4 GPU** before running. All build output goes to `build/`.

In [1]:
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
REPO = "tattack99/DD2360-assignment-2"
REPO_DIR = "/content/repo"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        from google.colab import userdata
        token = userdata.get("GH_TOKEN_DD2360_ASSIGNMENT_2")
        !git clone -q https://{token}@github.com/{REPO}.git {REPO_DIR}
    %cd {REPO_DIR}
    !git pull -q

/content/repo


In [2]:
!pwd

/content/repo


In [3]:
# Check that a GPU is attached and nvcc is available
!nvidia-smi
!nvcc --version

Wed Nov 18 19:22:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Q1 - 1D Convolution

In [4]:
# Quick check: both kernels should print PASS
!make run

nvcc -arch=native -DTILE_WIDTH=256 -o build/hw2-convolution_tile256 assignment/hw2-convolution.cu
./build/hw2-convolution_tile256 1024
Timing - Allocated host memory                                          5 us
Timing - Allocated device memory                                   307799 us
Timing - Initialize arrays with values                                 37 us
Timing - Copying data to the GPU                                      696 us
Timing - Finished 1D convolution(basic)                               629 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         56 us
Timing - Finished 1D convolution(tiled)                                45 us
P_tiled[0] = 1.233154
P_tiled[1] = 1.405636
P_tiled[2] = 1.397376
P_tiled[3] = 1.596605
P_tiled[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         41 us
Basic kernel vs CPU: PASS (0 mismatches)
Tiled kernel vs CP

In [5]:
# Q1.3 tile size: run every tile size on the largest input, compare the tiled kernel times
!make tilesweep ARGS=16384

=== TILE = 32 ===
nvcc -arch=native -DTILE_WIDTH=32 -o build/hw2-convolution_tile32 assignment/hw2-convolution.cu
./build/hw2-convolution_tile32 16384
Timing - Allocated host memory                                         11 us
Timing - Allocated device memory                                   205867 us
Timing - Initialize arrays with values                                420 us
Timing - Copying data to the GPU                                       91 us
Timing - Finished 1D convolution(basic)                               131 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         54 us
Timing - Finished 1D convolution(tiled)                                36 us
P_tiled[0] = 1.233154
P_tiled[1] = 1.405636
P_tiled[2] = 1.397376
P_tiled[3] = 1.596605
P_tiled[4] = 1.297674
Timing - Copying output P to the CPU and print out the results        112 us
Basic kernel vs CPU: PASS (0 mismatches)
Ti

In [6]:
# Q1.3 screenshot: N = 1024 ... 16384 with the chosen tile size
TILE = 256
!make sweep TILE={TILE}

=== N = 1024, TILE = 256 ===
Timing - Allocated host memory                                          5 us
Timing - Allocated device memory                                   172133 us
Timing - Initialize arrays with values                                 53 us
Timing - Copying data to the GPU                                       43 us
Timing - Finished 1D convolution(basic)                               139 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         40 us
Timing - Finished 1D convolution(tiled)                                33 us
P_tiled[0] = 1.233154
P_tiled[1] = 1.405636
P_tiled[2] = 1.397376
P_tiled[3] = 1.596605
P_tiled[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         32 us
Basic kernel vs CPU: PASS (0 mismatches)
Tiled kernel vs CPU: PASS (0 mismatches)
Timing - Free memory resources                                        121 us
=== N 

In [7]:
# Q1.4 Nsight Systems: where the time goes
!make profile TILE={TILE} ARGS=16384

nsys profile --stats=true --force-overwrite true -o build/hw2-convolution-report ./build/hw2-convolution_tile256 16384
Timing - Allocated host memory                                         12 us
Timing - Allocated device memory                                   475717 us
Timing - Initialize arrays with values                                445 us
Timing - Copying data to the GPU                                       77 us
Timing - Finished 1D convolution(basic)                              1061 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         67 us
Timing - Finished 1D convolution(tiled)                                75 us
P_tiled[0] = 1.233154
P_tiled[1] = 1.405636
P_tiled[2] = 1.397376
P_tiled[3] = 1.596605
P_tiled[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         92 us
Basic kernel vs CPU: PASS (0 mismatches)
Tiled kernel vs CPU: PASS (0 misma

In [8]:
# Q1.4 Nsight Compute: occupancy, shared memory, memory throughput
!make ncu TILE={TILE} ARGS=16384

ncu --section SpeedOfLight --section MemoryWorkloadAnalysis --section Occupancy --section LaunchStats ./build/hw2-convolution_tile256 16384
Timing - Allocated host memory                                          6 us
==PROF== Connected to process 1154 (/content/repo/build/hw2-convolution_tile256)
Timing - Allocated device memory                                   762542 us
Timing - Initialize arrays with values                                535 us
Timing - Copying data to the GPU                                       76 us
==PROF== Profiling "convolution_1D_basic" - 0: 0%....50%....100% - 9 passes
Timing - Finished 1D convolution(basic)                           3654882 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results        152 us
==PROF== Profiling "convolution_1D_tiled" - 1: 0%....50%....100% - 9 passes
Timing - Finished 1D convolution(tiled)                           1853318 us
P_tiled[

# Question 3

In [9]:
%cd /content/repo
%rm -rf llama2.c
!ls

/content/repo
assignment  build  colab.ipynb	Makefile  README.md


In [10]:
!git clone https://github.com/karpathy/llama2.c.git
%cd llama2.c

Cloning into 'llama2.c'...
remote: Enumerating objects: 1517, done.
remote: Counting objects: 100% (837/837), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 1517 (delta 813), reused 803 (delta 803), pack-reused 680 (from 1)
Receiving objects: 100% (1517/1517), 1.21 MiB | 11.70 MiB/s, done.
Resolving deltas: 100% (936/936), done.
/content/repo/llama2.c


In [12]:
#!make runfast

gcc -Ofast -o run run.c -lm
gcc -Ofast -o runq runq.c -lm


In [17]:
model = "stories15M" #We want the small model only
download_url = "https://huggingface.co/karpathy/tinyllamas/resolve/main/stories15M.bin"
!wget $download_url
model_file = model + ".bin"

--2026-11-18 19:27:16--  https://huggingface.co/karpathy/tinyllamas/resolve/main/stories15M.bin
Resolving huggingface.co (huggingface.co)... 99.86.101.56, 99.86.101.36, 99.86.101.64, ...
Connecting to huggingface.co (huggingface.co)|99.86.101.56|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/64c19eb0275dc8cd7f170b8f/07dbdd97d4def5722a8deaeb97073a464e041687313cbf6b06cccb9efdd0c4b7?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27stories15M.bin%3B+filename%3D%22stories15M.bin%22%3B&user_id=public&response-content-type=application%2Foctet-stream&xip=CEIgaS0hlXo&X-Xet-Cas-Uid=public&Expires=1795033636&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRjMTllYjAyNzVkYzhjZDdmMTcwYjhmLzA3ZGJkZDk3ZDRkZWY1NzIyYThkZWFlYjk3MDczYTQ2NGUwNDE2ODczMTNjYmY2YjA2Y2NjYjllZmRkMGM0YjdcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomdXNlcl9pZD1wdWJsaWMmcmVzcG9uc2UtY29udGVudC10eXBlPSom

In [18]:
#@title Generate Stories

# Generate args
max_token = 256 #@param {type:"slider", min:32, max:1024, step:32}
temperature = 0.8 #@param {type:"slider", min:0.0, max:1, step:0.05}
top_p = 0.9 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
prompt = "One day, Lily met a Shoggoth" #@param {type:"string"}

print(f"model: {model_file}, max_token: {max_token}, temperature: {temperature}, top_p: {top_p}, prompt: {prompt}")
print(f"----------------------------\n")

cmd = f'./run {model_file} -t {temperature} -p {top_p} -n {max_token} -i "{prompt}"'
!{cmd}

model: stories15M.bin, max_token: 256, temperature: 0.8, top_p: 0.9, prompt: One day, Lily met a Shoggoth
----------------------------

One day, Lily met a Shoggoth. She was very happy and asked her mom to take her to the store. "Mommy, can I have this shirt?" she asked. Her mom said yes and they went to the store together.
At the store, Lily saw a shirt with a blue hat. She asked her mom if she could have it. Her mom said yes and Lily was very happy. She put on the shirt and twirled around in it.
After they finished shopping, Lily and her mom went home. Lily's mom gave her a special surprise. She had bought a modern toy shirt for Lily to play with. Lily was so happy and thanked her mom. She played with her new toy all day and loved wearing it.
achieved tok/s: 128.433556
matmul calls: 7482


In [16]:
#@title Run Meta's Llama 2 models - Not sure we do this...

#@markdown input your huggingface [access token](https://huggingface.co/settings/tokens) to download Meta's Llama 2 models.

from huggingface_hub import snapshot_download

token = "replace your huggingface access token" #@param {type:"string"}
path = snapshot_download(repo_id="meta-llama/Llama-2-7b",cache_dir="Llama-2-7b", use_auth_token=token)

!python export.py llama2_7b.bin --meta-llama $path

print("./run llama2_7b.bin\n")
!./run llama2_7b.bin

TypeError: snapshot_download() got an unexpected keyword argument 'use_auth_token'